# DS Automation Assignment

# Task 1
Using our prepared churn data from week 2:
- use pycaret to find an ML algorithm that performs best on the data
    - Choose a metric you think is best to use for finding the best model; by default, it is accuracy but it could be AUC, precision, recall, etc. The week 3 FTE has some information on these different metrics.
- save the model to disk
- create a Python script/file/module with a function that takes a pandas dataframe as an input and returns the probability of churn for each row in the dataframe
    - your Python file/function should print out the predictions for new data (new_churn_data.csv)
    - the true values for the new data are [1, 0, 0, 1, 0] if you're interested
- test your Python module and function with the new data, new_churn_data.csv
- write a short summary of the process and results at the end of this notebook
- upload this Jupyter Notebook and Python file to a Github repository, and turn in a link to the repository in the week 5 assignment dropbox

Additional challenges:
- return the probability of churn for each new prediction, and the percentile where that prediction is in the distribution of probability predictions from the training dataset (e.g. a high probability of churn like 0.78 might be at the 90th percentile)
- use other autoML packages, such as TPOT, H2O, MLBox, etc, and compare performance and features with pycaret
- create a class in your Python module to hold the functions that you created
- accept user input to specify a file using a tool such as Python's `input()` function, the `click` package for command-line arguments, or a GUI
- Use the unmodified churn data (new_unmodified_churn_data.csv) in your Python script. This will require adding the same preprocessing steps from week 2 since this data is like the original unmodified dataset from week 1.


# Task 2: Critical Reflection
What are your observations based on the analyses you had performed? Is there one model that seems particularly useful for predicting churn. Why or why not?

# Summary

Write a short summary of the process and results here.

# Task 1



In [55]:
# Install PyCaret if not already installed
# !pip install --pre pycaret

## Import Libraries and Load the Prepared Churn Data from Week 2

We will use the `churn_data_encoded.csv` file that was created at the end of Week 2. This file contains the cleaned, feature-engineered, and encoded churn dataset with 7,032 rows and 13 columns.

In [56]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pycaret.classification import *
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
sns.set_style('whitegrid')

# Load the prepared churn data from Week 2
churn_df = pd.read_csv('churn_data_encoded.csv')

# Display the first few rows and info
print("Dataset shape:", churn_df.shape)
print("\nFirst 5 rows:")
print(churn_df.head())
print("\nData types:")
print(churn_df.dtypes)
print("\nMissing values:")
print(churn_df.isnull().sum())

Dataset shape: (7032, 13)

First 5 rows:
   customerID  tenure  PhoneService  Contract  PaymentMethod  MonthlyCharges  \
0        5365       1             0         0              2           29.85   
1        3953      34             1         1              3           56.95   
2        2558       2             1         0              3           53.85   
3        5524      45             0         1              0           42.30   
4        6500       2             1         0              2           70.70   

   TotalCharges  Churn  AvgMonthlyCharge  TenureMonthlyRatio  LifetimeValue  \
0         29.85      0         29.850000            0.033501       891.0225   
1       1889.50      0         55.573529            0.597015    107607.0250   
2        108.15      1         54.075000            0.037140      5823.8775   
3       1840.75      0         40.905556            1.063830     77863.7250   
4        151.65      1         75.825000            0.028289     10721.6550   

  T

## Setup PyCaret

We configure PyCaret with `Churn` as the target variable. We choose **AUC** as our optimization metric because:
- It handles class imbalance well (our churn rate is ~26.6%).
- It measures the model's ability to distinguish between classes across all classification thresholds.
- It is more informative than accuracy for imbalanced datasets.

We will drop `customerID` since it is an identifier with no predictive value. We also need to handle the categorical bin columns `TenureBin` and `ChargeBin` — PyCaret will automatically encode them.

In [57]:
churn_df_model = churn_df.drop(columns=['customerID'])


#   - The dataframe is passed to .fit().
#   - `silent`, `verbose`, `categorical_features`, and `ignore_features`
exp = ClassificationExperiment(
    target='Churn',
    session_id=42,
    train_size=0.8,
    normalize=True,
    log_experiment=False
).fit(churn_df_model)

print("Experiment fitted successfully!")
print("Target column:", exp.target)
print("Training shape:", (exp.X_train.shape[0], exp.X_train.shape[1] + 1))
print("Testing shape:", (exp.X_test.shape[0], exp.X_test.shape[1] + 1))

Experiment fitted successfully!
Target column: Churn
Training shape: (5625, 12)
Testing shape: (1407, 12)


## Compare Models

PyCaret compares multiple classification algorithms and ranks them by AUC. The top-performing model will be selected for tuning and deployment.

In [58]:
# Compare all models using AUC as the metric
compare_result = exp.compare_models(sort='AUC', n_select=1)
best_model = compare_result.best

print("Best model selected:")
print(best_model)
print("\nLeaderboard:")
print(compare_result.leaderboard)

[LightGBM] [Info] Number of positive: 1345, number of negative: 3717
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000351 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1370
[LightGBM] [Info] Number of data points in the train set: 5062, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265705 -> initscore=-1.016523
[LightGBM] [Info] Start training from score -1.016523
[LightGBM] [Info] Number of positive: 1345, number of negative: 3717
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000737 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1370
[LightGBM] [Info] Number of data points in the train set: 5062, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265705 -> initscore=-1.016523
[LightGB

## Tune the Best Model

We tune the best model to improve its AUC performance using PyCaret's built-in hyperparameter tuning.

In [59]:
# Tune the best model to improve performance
tuned_model = exp.tune_model(best_model, optimize='AUC')

print("Tuned model:")
print(tuned_model)

Tuned model:
TuneResult(pipeline=Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('numerical_pipeline',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['tenure', 'PhoneService',
                                                   'Contract', 'PaymentMethod',
                                                   'MonthlyCharges',
                                                   'TotalCharges',
                                                   'AvgMonthlyCharge',
                                                   'TenureMonthlyRatio',
                                                   'LifetimeValue']),
                             

## Evaluate the Tuned Model

We evaluate the tuned model using PyCaret's `evaluate_model` function, which generates multiple diagnostic plots including the confusion matrix, ROC curve, precision-recall curve, feature importance, and more.

In [60]:
# The tuned model's metrics are available directly
print("Tuned Model Metrics:")
print(tuned_model.metrics)

# Predict on the test set and view predictions
predictions = exp.predict_model(tuned_model.pipeline)
print("\nPredictions on test set:")
print(predictions.predictions.head())

Tuned Model Metrics:
        Accuracy     AUC  Recall   Prec.      F1   Kappa     MCC
Fold 0    0.7940  0.8536  0.5467  0.6308  0.5857  0.4495  0.4516
Fold 1    0.7869  0.8233  0.4600  0.6389  0.5349  0.4014  0.4105
Fold 2    0.7798  0.8301  0.4733  0.6121  0.5338  0.3927  0.3983
Fold 3    0.7886  0.8287  0.4733  0.6396  0.5441  0.4105  0.4183
Fold 4    0.7851  0.8454  0.4333  0.6436  0.5179  0.3864  0.3989
Fold 5    0.7972  0.8416  0.5369  0.6400  0.5839  0.4512  0.4542
Fold 6    0.8025  0.8500  0.5302  0.6583  0.5874  0.4595  0.4642
Fold 7    0.7811  0.8447  0.5235  0.6000  0.5591  0.4145  0.4162
Fold 8    0.7722  0.8308  0.4564  0.5913  0.5152  0.3695  0.3748
Fold 9    0.8149  0.8658  0.4497  0.7528  0.5630  0.4549  0.4793
Mean      0.7902  0.8414  0.4883  0.6407  0.5525  0.4190  0.4266
Std       0.0117  0.0126  0.0394  0.0423  0.0260  0.0308  0.0320

Predictions on test set:
      tenure  PhoneService  Contract  PaymentMethod  MonthlyCharges  \
971       59             1         2 

In [61]:
# Print the final model performance metrics on the hold-out test set
final_results = exp.pull()
print("Final Model Performance Metrics:")
print(final_results)

Final Model Performance Metrics:
        Accuracy     AUC  Recall   Prec.      F1   Kappa     MCC
Fold 0    0.7940  0.8536  0.5467  0.6308  0.5857  0.4495  0.4516
Fold 1    0.7869  0.8233  0.4600  0.6389  0.5349  0.4014  0.4105
Fold 2    0.7798  0.8301  0.4733  0.6121  0.5338  0.3927  0.3983
Fold 3    0.7886  0.8287  0.4733  0.6396  0.5441  0.4105  0.4183
Fold 4    0.7851  0.8454  0.4333  0.6436  0.5179  0.3864  0.3989
Fold 5    0.7972  0.8416  0.5369  0.6400  0.5839  0.4512  0.4542
Fold 6    0.8025  0.8500  0.5302  0.6583  0.5874  0.4595  0.4642
Fold 7    0.7811  0.8447  0.5235  0.6000  0.5591  0.4145  0.4162
Fold 8    0.7722  0.8308  0.4564  0.5913  0.5152  0.3695  0.3748
Fold 9    0.8149  0.8658  0.4497  0.7528  0.5630  0.4549  0.4793
Mean      0.7902  0.8414  0.4883  0.6407  0.5525  0.4190  0.4266
Std       0.0117  0.0126  0.0394  0.0423  0.0260  0.0308  0.0320


In [62]:
from sklearn.metrics import classification_report, roc_auc_score

# Get test data and predictions
y_test = exp.y_test

# Get the prediction dataframe
pred_df = predictions.predictions

# In PyCaret 4.0, the label column may be named 'prediction_label' or 'prediction'
if 'prediction_label' in pred_df.columns:
    y_pred = pred_df['prediction_label']
elif 'prediction' in pred_df.columns:
    y_pred = pred_df['prediction']
else:
    # Fallback: use the last column
    y_pred = pred_df.iloc[:, -1]

# Get the probability column
if 'prediction_score' in pred_df.columns:
    y_proba = pred_df['prediction_score']
elif 'Score' in pred_df.columns:
    y_proba = pred_df['Score']
else:
    # Fallback: second-to-last column
    y_proba = pred_df.iloc[:, -2]

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

# Print ROC-AUC
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")

Classification Report:
              precision    recall  f1-score   support

    No Churn       0.82      0.91      0.86      1033
       Churn       0.64      0.46      0.53       374

    accuracy                           0.79      1407
   macro avg       0.73      0.68      0.70      1407
weighted avg       0.77      0.79      0.77      1407

ROC-AUC: 0.8296


## Save the Model to Disk

The tuned model pipeline is saved to disk using `joblib.dump()`. The saved file is `churn_model.joblib`.

Saving the complete pipeline preserves the preprocessing and trained model so that the same pipeline can be loaded later for predictions on compatible input data.

In [63]:
import joblib

# Save the tuned model's pipeline using joblib
joblib.dump(tuned_model.pipeline, 'churn_model.joblib')
print("Model saved as 'churn_model.joblib'")

Model saved as 'churn_model.joblib'


## Create the Python Module

The Python module `churn_predictor.py` contains a `ChurnPredictor` class for loading the saved churn model and generating predictions.

The class:

- Loads the saved `churn_model.joblib` pipeline.
- Accepts a pandas DataFrame containing features compatible with the trained model.
- Calculates the probability of churn for each row.
- Generates a binary churn prediction using a 0.5 probability threshold.
- Calculates the percentile of each churn probability relative to the training probability distribution.
- Prints the prediction results in a readable format.

The module is also designed to be executable as a standalone Python script using `new_churn_data.csv`.

In [64]:
module_code = '''
import pandas as pd
import numpy as np
import joblib
import os


class ChurnPredictor:
    """
    Load a trained churn model pipeline and generate churn predictions.
    """

    def __init__(self, model_path='churn_model.joblib'):
        """
        Initialize the predictor by loading the saved model pipeline.
        """
        self.pipeline = joblib.load(model_path)
        self.training_probs = None

    def set_training_probs(self, training_probs):
        """
        Store training probability predictions for percentile calculation.
        """
        self.training_probs = np.asarray(training_probs)

    @staticmethod
    def prepare_data(df):
        """
        Prepare Week 2-style churn data for prediction.
        """

        df = df.copy()

        # Convert TotalCharges to numeric if necessary
        if 'TotalCharges' in df.columns:
            df['TotalCharges'] = pd.to_numeric(
                df['TotalCharges'],
                errors='coerce'
            )

        # Remove identifier
        if 'customerID' in df.columns:
            df = df.drop(columns=['customerID'])

        # Create engineered numerical features
        if 'AvgMonthlyCharge' not in df.columns:
            df['AvgMonthlyCharge'] = (
                df['TotalCharges'] /
                df['tenure'].replace(0, np.nan)
            )

        if 'TenureMonthlyRatio' not in df.columns:
            df['TenureMonthlyRatio'] = (
                df['tenure'] /
                df['MonthlyCharges'].replace(0, np.nan)
            )

        if 'LifetimeValue' not in df.columns:
            df['LifetimeValue'] = (
                df['MonthlyCharges'] *
                df['tenure']
            )

        # Create categorical tenure bins
        if 'TenureBin' not in df.columns:
            df['TenureBin'] = pd.cut(
                df['tenure'],
                bins=[-1, 6, 12, 24, 48, 72],
                labels=[
                    '0-6',
                    '6-12',
                    '12-24',
                    '24-48',
                    '48-72'
                ]
            )

        # Create categorical charge bins
        if 'ChargeBin' not in df.columns:
            df['ChargeBin'] = pd.cut(
                df['MonthlyCharges'],
                bins=[0, 30, 50, 70, 90, 120],
                labels=[
                    '0-30',
                    '30-50',
                    '50-70',
                    '70-90',
                    '90-120'
                ]
            )

        # Remove columns that were not used by the trained model
        if 'charge_per_tenure' in df.columns:
            df = df.drop(columns=['charge_per_tenure'])

        # Remove target if supplied
        if 'Churn' in df.columns:
            df = df.drop(columns=['Churn'])

        # Required model feature order
        expected_cols = [
            'tenure',
            'PhoneService',
            'Contract',
            'PaymentMethod',
            'MonthlyCharges',
            'TotalCharges',
            'AvgMonthlyCharge',
            'TenureMonthlyRatio',
            'LifetimeValue',
            'TenureBin',
            'ChargeBin'
        ]

        # Check that all required columns exist
        missing_cols = [
            col for col in expected_cols
            if col not in df.columns
        ]

        if missing_cols:
            raise ValueError(
                f"Missing required model features: {missing_cols}"
            )

        return df[expected_cols]

    def predict(self, df, prepare=True):
        """
        Generate churn probabilities and predictions.
        """

        if prepare:
            model_data = self.prepare_data(df)
        else:
            model_data = df.copy()

        # Generate churn probabilities
        churn_prob = self.pipeline.predict_proba(
            model_data
        )[:, 1]

        results = df.copy()

        results['Churn_Probability'] = churn_prob

        results['Churn_Prediction'] = (
            results['Churn_Probability'] >= 0.5
        ).astype(int)

        # Calculate percentile relative to training probabilities
        if self.training_probs is not None:
            results['Churn_Percentile'] = (
                results['Churn_Probability']
                .apply(
                    lambda x:
                    (self.training_probs < x).mean() * 100
                )
            )

        return results

    def predict_file(self, file_path):
        """
        Load a CSV file and generate churn predictions.

        Parameters
        ----------
        file_path : str
            Path to the CSV file.

        Returns
        -------
        pandas.DataFrame
            Prediction results.
        """

        if not os.path.exists(file_path):
            raise FileNotFoundError(
                f"File not found: {file_path}"
            )

        data = pd.read_csv(file_path)

        print(f"Loaded file: {file_path}")
        print(f"Rows: {len(data)}")
        print(f"Columns: {len(data.columns)}")

        return self.predict(data, prepare=True)

    def print_predictions(self, df, prepare=True):
        """
        Print prediction results for a DataFrame.
        """

        results = self.predict(
            df,
            prepare=prepare
        )

        print("\\nChurn Predictions:")
        print(
            results[
                [
                    'Churn_Probability',
                    'Churn_Prediction'
                ]
            ]
        )

        if 'Churn_Percentile' in results.columns:
            print(
                "\\nChurn Percentiles "
                "(relative to training distribution):"
            )

            print(
                results[
                    [
                        'Churn_Probability',
                        'Churn_Percentile'
                    ]
                ]
            )

        return results

    def predict_file_interactive(self):
        """
        Ask the user for a CSV file path and generate predictions.
        """

        file_path = input(
            "Enter the path to the churn CSV file "
            "[default: new_churn_data.csv]: "
        ).strip()

        if not file_path:
            file_path = 'new_churn_data.csv'

        results = self.predict_file(file_path)

        self.print_predictions_from_results(results)

        return results

    @staticmethod
    def print_predictions_from_results(results):
        """
        Print an existing prediction DataFrame.
        """

        print("\\nPrediction Results:")
        print(
            results[
                [
                    'Churn_Probability',
                    'Churn_Prediction'
                ]
            ]
        )

        if 'Churn_Percentile' in results.columns:
            print(
                "\\nChurn Percentiles:"
            )

            print(
                results[
                    [
                        'Churn_Probability',
                        'Churn_Percentile'
                    ]
                ]
            )


if __name__ == '__main__':

    predictor = ChurnPredictor(
        'churn_model.joblib'
    )

    # Allow the user to specify the input CSV file
    file_path = input(
        "Enter the path to the churn CSV file "
        "[default: new_churn_data.csv]: "
    ).strip()

    if not file_path:
        file_path = 'new_churn_data.csv'

    results = predictor.predict_file(
        file_path
    )

    predictor.print_predictions_from_results(
        results
    )
'''

with open('churn_predictor.py', 'w') as f:
    f.write(module_code)

print("Updated churn_predictor.py successfully.")

Updated churn_predictor.py successfully.


## Test the Python Module with New Data

The Python module is tested using `new_churn_data.csv`.

The raw new data does not contain the `Churn` target variable and does not initially contain all of the engineered features used by the trained model. The `ChurnPredictor` class therefore performs the required preprocessing internally, including:

- Removing `customerID`.
- Creating `AvgMonthlyCharge`.
- Creating `TenureMonthlyRatio`.
- Creating `LifetimeValue`.
- Creating `TenureBin`.
- Creating `ChargeBin`.
- Removing `charge_per_tenure`, which was not used during model training.
- Reordering the columns to match the trained model's feature schema.

This allows the Python module to be tested using the original `new_churn_data.csv` without requiring separate preprocessing code in the notebook.

In [65]:
# Import the module we created
import importlib
import churn_predictor

importlib.reload(churn_predictor)

from churn_predictor import ChurnPredictor

# Load the raw new churn data
new_data = pd.read_csv('new_churn_data.csv')

print("New data shape:", new_data.shape)
print("\nNew data columns:")
print(new_data.columns.tolist())

print("\nNew data:")
display(new_data)

# Initialize the predictor
predictor = ChurnPredictor('churn_model.joblib')

# Generate training probabilities for percentile calculation
train_probs = tuned_model.pipeline.predict_proba(exp.X_train)[:, 1]

predictor.set_training_probs(train_probs)

# Generate predictions.
# The module performs the required preprocessing automatically.
results = predictor.print_predictions(new_data)

print("\nFull Results:")
display(results)

New data shape: (5, 8)

New data columns:
['customerID', 'tenure', 'PhoneService', 'Contract', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'charge_per_tenure']

New data:


,customerID,tenure,PhoneService,Contract,PaymentMethod,MonthlyCharges,TotalCharges,charge_per_tenure
0,9305-CKSKC,22,1,0,2,97.40,811.70,36.895455
1,1452-KNGVK,8,0,1,1,77.30,1701.95,212.743750
2,6723-OKKJM,28,1,0,0,28.25,250.90,8.960714
3,7832-POPKP,62,1,0,2,101.70,3106.56,50.105806
4,6348-TACGU,10,0,0,1,51.15,3440.97,344.097000



Churn Predictions:
   Churn_Probability  Churn_Prediction
0           0.451574                 0
1           0.189215                 0
2           0.043248                 0
3           0.253950                 0
4           0.274610                 0

Churn Percentiles (relative to training distribution):
   Churn_Probability  Churn_Percentile
0           0.451574         76.782222
1           0.189215         51.431111
2           0.043248         21.920000
3           0.253950         58.275556
4           0.274610         60.106667

Full Results:


,customerID,tenure,PhoneService,Contract,PaymentMethod,MonthlyCharges,TotalCharges,charge_per_tenure,Churn_Probability,Churn_Prediction,Churn_Percentile
0,9305-CKSKC,22,1,0,2,97.40,811.70,36.895455,0.451574,0,76.782222
1,1452-KNGVK,8,0,1,1,77.30,1701.95,212.743750,0.189215,0,51.431111
2,6723-OKKJM,28,1,0,0,28.25,250.90,8.960714,0.043248,0,21.920000
3,7832-POPKP,62,1,0,2,101.70,3106.56,50.105806,0.253950,0,58.275556
4,6348-TACGU,10,0,0,1,51.15,3440.97,344.097000,0.274610,0,60.106667


### Module Test Result

The Python module successfully loaded the saved `churn_model.joblib` pipeline, prepared the raw `new_churn_data.csv` data, generated churn probabilities, produced binary churn predictions, and calculated the percentile of each prediction relative to the training probability distribution.

The known true values for the five new customers are:

`[1, 0, 0, 1, 0]`

These values are used in the following cell to compare the model predictions with the known outcomes.

## Compare Predictions with the Known True Values

The assignment provides the true churn values for the five records in `new_churn_data.csv` as:

`[1, 0, 0, 1, 0]`

The predicted values are compared with these known values. Because this is only a five-record sample, the resulting accuracy should not be interpreted as the overall performance of the trained model. The larger hold-out test-set results provide a more reliable evaluation of model performance.

In [66]:
# True values for the new data
true_values = [1, 0, 0, 1, 0]

predicted_values = results['Churn_Prediction'].tolist()

print("True values:     ", true_values)
print("Predicted values:", predicted_values)
print("Match:", true_values == predicted_values)

from sklearn.metrics import accuracy_score, classification_report

print("\nAccuracy:", accuracy_score(true_values, predicted_values))

print("\nClassification Report:")
print(
    classification_report(
        true_values,
        predicted_values,
        target_names=['No Churn', 'Churn'],
        zero_division=0
    )
)

True values:      [1, 0, 0, 1, 0]
Predicted values: [0, 0, 0, 0, 0]
Match: False

Accuracy: 0.6

Classification Report:
              precision    recall  f1-score   support

    No Churn       0.60      1.00      0.75         3
       Churn       0.00      0.00      0.00         2

    accuracy                           0.60         5
   macro avg       0.30      0.50      0.38         5
weighted avg       0.36      0.60      0.45         5



# Task 2: Critical Reflection

## Observations Based on the Analyses

### Observation 1: PyCaret Selected Gradient Boosting

PyCaret compared multiple classification algorithms using **AUC** as the evaluation metric.

The model comparison selected the **Gradient Boosting Classifier (`gbc`)** as the top-performing model in the comparison. Its AUC in the PyCaret comparison was approximately **0.8423**.

This provides an objective result from the Week 5 experiment rather than relying on an expected or assumed model ranking.

### Observation 2: Hyperparameter Tuning Produced a Similar Cross-Validation AUC

The selected Gradient Boosting model was tuned using:

`exp.tune_model(best_model, optimize='AUC')`

The tuned model achieved a mean cross-validation AUC of approximately **0.8414** during the tuning process.

The hold-out test-set evaluation produced a ROC-AUC of approximately **0.8296**.

These values represent different evaluations:

- **0.8423** — AUC from the PyCaret model comparison.
- **0.8414** — mean cross-validation AUC for the tuned model.
- **0.8296** — ROC-AUC on the independent hold-out test set.

The difference between cross-validation and hold-out performance is expected because they are calculated on different data partitions.

### Observation 3: AUC Is Appropriate for the Churn Dataset

The churn dataset is imbalanced, with approximately **73.42% No Churn** and **26.58% Churn**.

A model that always predicted No Churn would therefore achieve approximately 73.42% accuracy. Accuracy alone does not describe how well the model identifies customers who actually churn.

AUC is useful because it measures the model's ability to distinguish between churn and non-churn cases across different classification thresholds.

For this reason, AUC was selected as the primary metric for the Week 5 PyCaret model comparison and tuning.

### Observation 4: The Model Identifies Non-Churn Customers More Reliably Than Churn Customers

On the hold-out test set, the tuned model achieved approximately:

- Accuracy: **79%**
- ROC-AUC: **0.8296**
- No Churn precision: **0.82**
- No Churn recall: **0.91**
- Churn precision: **0.64**
- Churn recall: **0.46**
- Churn F1-score: **0.53**

The churn recall of 0.46 indicates that the model did not identify all actual churn cases at the default 0.5 classification threshold.

This is an important limitation. The probability output can therefore be useful for ranking or prioritizing customers for further analysis, but the binary prediction should not be interpreted as a perfect identification of future churn.


### Observation 5: Comparison with a Second AutoML Framework

A second AutoML framework, **H2O AutoML**, was used as an additional challenge.

H2O AutoML automatically trained and compared multiple machine learning models using AUC as the primary metric. Its leaderboard and hold-out test performance were compared with the PyCaret workflow.

The comparison is useful because different AutoML frameworks search different algorithms, hyperparameter configurations, and ensemble strategies. Therefore, the best-performing model from one framework does not necessarily have the same performance as the best-performing model from another framework.

The actual H2O AutoML leaderboard and hold-out ROC-AUC reported in this notebook should be used when discussing the numerical difference between the two frameworks.

### Observation 6: The Prediction Module Can Accept Different Input Files

The `ChurnPredictor` module was extended to support user-specified CSV files.

When the module is executed directly, the user is prompted:

`Enter the path to the churn CSV file [default: new_churn_data.csv]:`

This makes the prediction workflow reusable for different customer datasets instead of requiring the input filename to be hard-coded.

### Observation 7: The Model Can Process Unmodified Week 1 Data

The module and notebook were extended to process `new_unmodified_churn_data.csv`.

The Week 2 preprocessing workflow was recreated for this data, including:

- Converting `TotalCharges` to numeric.
- Removing rows with invalid or missing `TotalCharges`.
- Creating `AvgMonthlyCharge`.
- Creating `TenureMonthlyRatio`.
- Creating `LifetimeValue`.
- Creating `TenureBin`.
- Creating `ChargeBin`.
- Encoding `PhoneService`.
- Encoding `Contract`.
- Encoding `PaymentMethod`.
- Removing `customerID`.
- Removing the target variable before prediction.
- Aligning the final features with the saved model's expected schema.

This makes the trained Week 5 model usable with data closer to the original Week 1 format rather than requiring an already encoded Week 2 dataset.

## Is There One Model That Seems Particularly Useful?

The Week 5 experiment provides evidence that the **Gradient Boosting Classifier** performed strongly in the PyCaret model comparison when AUC was used as the metric.

However, model usefulness depends on the evaluation criterion and the intended application.

For example:

- **AUC** evaluates overall discrimination across classification thresholds.
- **Recall** is important when missing a churn customer has a high cost.
- **Precision** is important when retention resources are limited and false positives are costly.
- **Interpretability** may be important when model decisions need to be explained.
- **Deployment requirements** may favor a model that is easier to maintain and integrate.

Therefore, the Week 5 results support using the Gradient Boosting model as the selected model for this assignment, while also showing that its performance should be evaluated using more than one metric.

### Overall Reflection

The Week 5 PyCaret workflow successfully automated model comparison, selected a Gradient Boosting Classifier based on AUC, tuned the selected model, saved the resulting pipeline, and integrated it into a Python prediction module.

The hold-out ROC-AUC of **0.8296** indicates useful discriminatory performance, while the **0.46 recall for the churn class** shows that there is still room for improvement in identifying actual churn cases.

Future work could investigate threshold optimization, cost-sensitive learning, additional feature engineering, and comparison with other AutoML frameworks. These approaches could be evaluated using the same hold-out evaluation methodology.

# AI Use Disclosure

## How I Used AI

1. **PyCaret setup guidance**: Used AI assistance to understand how to configure PyCaret for the churn dataset, including selecting the target variable and evaluation metric.

2. **Model comparison and tuning**: Used AI assistance to understand how to use `compare_models(sort='AUC')` and `tune_model()` to compare classification models and tune the selected model.

3. **Model saving**: Used AI assistance to structure the process for saving the trained model pipeline with `joblib`.

4. **Python module creation**: Used AI assistance to structure the `churn_predictor.py` module and the `ChurnPredictor` class for loading the saved model and generating predictions.

5. **Probability and percentile calculation**: Used AI assistance to implement churn probability predictions and calculate the percentile of each prediction relative to the training probability distribution.

6. **Feature alignment**: Used AI assistance to identify and create the engineered features required by the trained model when processing `new_churn_data.csv`.

7. **Error debugging**: Used AI assistance to troubleshoot differences in PyCaret prediction column names and to make the prediction code more robust.

8. **Critical reflection**: Used AI assistance to organize and interpret the model evaluation results from the different weeks while ensuring that the final discussion reflects the actual results obtained.

## Something the AI Got Wrong or Had to Be Corrected

The initial prediction approach assumed that `new_churn_data.csv` contained the same engineered features as the Week 2 training dataset. However, the new data did not contain:

- `AvgMonthlyCharge`
- `TenureMonthlyRatio`
- `LifetimeValue`
- `TenureBin`
- `ChargeBin`

The new data instead contained the original input features together with `charge_per_tenure`.

The prediction workflow therefore required the same feature-engineering steps used during Week 2.

The feature preparation was subsequently incorporated into the `ChurnPredictor` module so that the Python module can process the raw `new_churn_data.csv` directly.

Another correction involved the model-saving description. The final implementation uses `joblib.dump()` to create `churn_model.joblib`, rather than PyCaret's `save_model()` function and a `.pkl` file.

## What I Verified

1. **Feature Alignment**: Verified that the features supplied to the trained model match the model's expected feature schema.

2. **Feature Engineering**: Verified the creation of:
   - `AvgMonthlyCharge`
   - `TenureMonthlyRatio`
   - `LifetimeValue`
   - `TenureBin`
   - `ChargeBin`

3. **Bin Boundaries**: Verified that the Week 2 bin boundaries are used:
   - `TenureBin`: `0-6`, `6-12`, `12-24`, `24-48`, `48-72`
   - `ChargeBin`: `0-30`, `30-50`, `50-70`, `70-90`, `90-120`

4. **Target Distribution**: Verified the approximate target distribution of 73.42% No Churn and 26.58% Churn.

5. **Known True Values**: Verified the five known values for `new_churn_data.csv` as `[1, 0, 0, 1, 0]`.

6. **Model Performance**: Verified the Week 5 hold-out ROC-AUC of approximately 0.8296 and the classification metrics produced by the tuned model.

7. **Model Selection**: Verified that PyCaret selected the Gradient Boosting Classifier as the top model in the AUC comparison, with an AUC of approximately 0.8423.

# Summary

## Overview

This notebook completed the **DS Automation** phase of the churn prediction project using **PyCaret**. The prepared Week 2 churn dataset was used to compare classification algorithms, select a model using AUC, tune the selected model, save the trained pipeline, and create a Python module for generating churn predictions on new customer data.

All required datasets and model files used in this assignment are available in the Google Colab environment used for the project.

## Key Activities Performed

### 1. Data Preparation

- Loaded the prepared Week 2 dataset, `churn_data_encoded.csv`.
- Used the cleaned and feature-engineered dataset for model development.
- Removed `customerID` because it is an identifier rather than a predictive feature.
- Used the engineered features required by the Week 2 dataset.

### 2. PyCaret Setup

PyCaret was configured with:

- `target='Churn'`
- `train_size=0.8`
- `session_id=42`
- `normalize=True`
- **AUC** as the model comparison and tuning metric.

AUC was selected because the churn target is imbalanced, with approximately **73.42% No Churn** and **26.58% Churn**.

### 3. Model Comparison

The PyCaret `compare_models()` function was used with AUC as the sorting metric.

The **Gradient Boosting Classifier (`gbc`)** was selected as the top-performing model in the comparison, with an AUC of approximately **0.8423**.

### 4. Hyperparameter Tuning

The selected Gradient Boosting model was tuned using:

`tune_model(best_model, optimize='AUC')`

The tuned model achieved a mean cross-validation AUC of approximately **0.8414**.

On the independent hold-out test set, the tuned model achieved:

- Accuracy: approximately **79%**
- ROC-AUC: approximately **0.8296**
- Churn precision: **0.64**
- Churn recall: **0.46**
- Churn F1-score: **0.53**

The difference between the cross-validation AUC and hold-out ROC-AUC reflects the fact that they are calculated using different evaluation procedures and data partitions.

### 5. Model Saving

The tuned model pipeline was saved using `joblib.dump()` as:

`churn_model.joblib`

The saved pipeline can therefore be loaded later for prediction without retraining the model.

### 6. Python Module

A Python module named `churn_predictor.py` was created.

The module contains the `ChurnPredictor` class, which:

- Loads the saved `churn_model.joblib` pipeline.
- Accepts churn data as a pandas DataFrame.
- Performs the required feature preparation.
- Generates churn probabilities.
- Generates binary churn predictions.
- Calculates churn probability percentiles relative to the training probability distribution.
- Prints the prediction results.

### 7. Testing with New Data

The module was tested using `new_churn_data.csv`.

The module prepares the raw data by:

- Removing `customerID`.
- Creating `AvgMonthlyCharge`.
- Creating `TenureMonthlyRatio`.
- Creating `LifetimeValue`.
- Creating `TenureBin`.
- Creating `ChargeBin`.
- Removing `charge_per_tenure`, which was not used during model training.
- Ordering the features according to the trained model's expected schema.

The known true values for the five new records were:

`[1, 0, 0, 1, 0]`

The model predictions were compared with these known values. Because this contains only five observations, the resulting accuracy is treated as a small validation example rather than as the overall model performance.

### 8. Additional Challenges Completed

The optional challenges were also implemented.

#### Additional AutoML Package

**H2O AutoML** was installed and used as a second AutoML framework.

H2O AutoML:

- Trained multiple candidate models.
- Ranked models using AUC.
- Produced an AutoML leaderboard.
- Generated predictions on the hold-out test set.
- Calculated a hold-out ROC-AUC.
- Was compared with the PyCaret tuned model.
- Provided feature importance where supported by the selected H2O model.

#### Python Class

The `ChurnPredictor` class was implemented in `churn_predictor.py`.

The class handles:

- Model loading.
- Feature preparation.
- Churn probability prediction.
- Binary churn prediction.
- Probability percentile calculation.
- CSV file prediction.

#### User File Input

The prediction module now accepts a user-specified CSV file.

When `churn_predictor.py` is executed directly, the user can enter the path to the input CSV file. If no path is supplied, `new_churn_data.csv` is used as the default.

#### Unmodified Week 1 Data

The optional `new_unmodified_churn_data.csv` challenge was implemented.

The Week 2 preprocessing workflow was recreated so that the original-style data can be transformed into the feature schema expected by the saved Week 5 model.

The preprocessing includes data cleaning, numerical feature engineering, categorical encoding, bin creation, removal of identifiers, and feature alignment.

## Final Results

The Week 5 workflow successfully:

1. Loaded the prepared Week 2 churn dataset.
2. Configured PyCaret for binary churn classification.
3. Compared multiple classification algorithms using AUC.
4. Selected the Gradient Boosting Classifier as the top model in the comparison.
5. Tuned the selected model.
6. Evaluated the tuned model on the hold-out test set.
7. Saved the trained pipeline as `churn_model.joblib`.
8. Created a reusable `ChurnPredictor` Python class.
9. Generated churn probabilities and binary predictions.
10. Calculated probability percentiles relative to the training probability distribution.
11. Tested the module with `new_churn_data.csv`.
12. Compared the predictions with the known true values.

## Critical Reflection

The Week 5 results show that the Gradient Boosting Classifier performed strongly in the PyCaret AUC comparison. The hold-out evaluation also demonstrated useful discriminatory performance, with a ROC-AUC of approximately **0.8296**.

However, the churn-class recall was **0.46**, showing that the model did not identify all actual churn cases at the default 0.5 classification threshold. This indicates that model evaluation should consider more than accuracy alone.

Further work could investigate alternative probability thresholds, cost-sensitive learning, additional feature engineering, and comparisons with other AutoML frameworks. These approaches could be evaluated using the same hold-out test methodology.

## Additional Challenge: Compare PyCaret with H2O AutoML

To extend the automated machine learning analysis, H2O AutoML is used as a second AutoML framework.

The H2O AutoML experiment uses the same prepared churn dataset and the same training/test partitions used by the PyCaret experiment where possible. AUC is retained as the primary comparison metric because the churn target is imbalanced.

The H2O AutoML workflow will:

1. Prepare the training and testing data for H2O.
2. Train multiple candidate models automatically.
3. Rank the models using AUC.
4. Display the H2O AutoML leaderboard.
5. Evaluate the selected H2O model on the hold-out test data.
6. Compare its AUC with the tuned PyCaret model.
7. Examine feature importance where supported by the selected H2O model.

In [67]:
# Install H2O AutoML
!pip install -q h2o

## Import and prepare H2O data

In [68]:
import h2o
from h2o.automl import H2OAutoML
from sklearn.metrics import roc_auc_score

# Initialize H2O
h2o.init()

# Use the same PyCaret train/test partitions
X_train_h2o = exp.X_train.copy()
X_test_h2o = exp.X_test.copy()

y_train_h2o = exp.y_train.copy()
y_test_h2o = exp.y_test.copy()

# Add target column
train_h2o_df = X_train_h2o.copy()
train_h2o_df['Churn'] = y_train_h2o.values

test_h2o_df = X_test_h2o.copy()
test_h2o_df['Churn'] = y_test_h2o.values

# Convert pandas DataFrames to H2OFrames
train_h2o = h2o.H2OFrame(train_h2o_df)
test_h2o = h2o.H2OFrame(test_h2o_df)

# Convert target to categorical for binary classification
train_h2o['Churn'] = train_h2o['Churn'].asfactor()
test_h2o['Churn'] = test_h2o['Churn'].asfactor()

# Convert categorical feature columns to factors
categorical_columns = ['TenureBin', 'ChargeBin']

for col in categorical_columns:
    train_h2o[col] = train_h2o[col].asfactor()
    test_h2o[col] = test_h2o[col].asfactor()

print("H2O training shape:", train_h2o.shape)
print("H2O testing shape:", test_h2o.shape)

Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,23 mins 12 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.12
H2O_cluster_version_age:,1 month and 16 days
H2O_cluster_name:,H2O_from_python_unknownUser_yn8qne
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.090 Gb
H2O_cluster_total_cores:,2
H2O_cluster_allowed_cores:,2
H2O_cluster_status:,"locked, healthy"



+------------------------------------------------------------------+
| You are running the community edition of H2O-3 OSS.              |
|                                                                  |
| For commercial use, H2O-3 Secure is now recommended.             |
| This includes production support, CVE fixes, multi-node scaling, |
| model artifact extraction, and more.                             |
| See h2o.ai/h2o-3/oss-vs-secure for additional details.           |
| Contact enterprise@h2o.ai to upgrade.                            |
+------------------------------------------------------------------+
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
H2O training shape: (5625, 12)
H2O testing shape: (1407, 12)


### Run H2O AutoML

H2O AutoML is configured to optimize **AUC**, matching the primary metric used in the PyCaret analysis.

The experiment is limited to a fixed number of models so that the analysis remains reproducible and practical in Google Colab.

In [69]:
# Define predictors and target
target = 'Churn'
features = [col for col in train_h2o.columns if col != target]

# Run H2O AutoML
h2o_aml = H2OAutoML(
    max_models=10,
    seed=42,
    sort_metric='AUC',
    verbosity='info'
)

h2o_aml.train(
    x=features,
    y=target,
    training_frame=train_h2o
)

# Display the H2O AutoML leaderboard
h2o_leaderboard = h2o_aml.leaderboard.as_data_frame()

print("H2O AutoML Leaderboard:")
display(h2o_leaderboard)

AutoML progress: |
11:01:38.444: Project: AutoML_2_20260928_110138
11:01:38.445: 5-fold cross-validation will be used.
11:01:38.460: Setting stopping tolerance adaptively based on the training frame: 0.013333333333333334
11:01:38.460: Build control seed: 42
11:01:38.461: training frame: Frame key: AutoML_2_20260928_110138_training_py_13_sid_ac01    cols: 12    rows: 5625  chunks: 1    size: 201261  checksum: -4070324912323109376
11:01:38.461: validation frame: NULL
11:01:38.461: leaderboard frame: NULL
11:01:38.461: blending frame: NULL
11:01:38.461: response column: Churn
11:01:38.461: fold column: null
11:01:38.461: weights column: null
11:01:38.466: Loading execution steps: [{XGBoost : [def_2 (1g, 10w), def_1 (2g, 10w), def_3 (3g, 10w), grid_1 (4g, 90w), lr_search (7g, 30w)]}, {GLM : [def_1 (1g, 10w)]}, {DRF : [def_1 (2g, 10w), XRT (3g, 10w)]}, {GBM : [def_5 (1g, 10w), def_2 (2g, 10w), def_3 (2g, 10w), def_4 (2g, 10w), def_1 (3g, 10w), grid_1 (4g, 60w), lr_annealing (7g, 10w)]}, {De

,model_id,auc,logloss,aucpr,mean_per_class_error,rmse,mse
0,StackedEnsemble_BestOfFamily_1_AutoML_2_202609...,0.842621,0.419200,0.640749,0.245868,0.369935,0.136852
1,StackedEnsemble_AllModels_1_AutoML_2_20260928_...,0.841852,0.419953,0.638477,0.242346,0.370361,0.137167
2,GBM_1_AutoML_2_20260928_110138,0.839531,0.422987,0.635862,0.234840,0.371430,0.137960
3,GLM_1_AutoML_2_20260928_110138,0.838231,0.424699,0.629497,0.244599,0.372526,0.138775
4,GBM_2_AutoML_2_20260928_110138,0.836540,0.427459,0.632247,0.241167,0.374061,0.139922
5,XGBoost_3_AutoML_2_20260928_110138,0.833157,0.432761,0.628215,0.243390,0.376741,0.141934
6,GBM_3_AutoML_2_20260928_110138,0.830226,0.434507,0.628937,0.241945,0.376802,0.141980
7,XGBoost_1_AutoML_2_20260928_110138,0.828362,0.440653,0.621086,0.254067,0.379592,0.144090
8,GBM_4_AutoML_2_20260928_110138,0.826507,0.439618,0.621455,0.246120,0.379275,0.143850
9,XGBoost_2_AutoML_2_20260928_110138,0.821943,0.454472,0.605831,0.250558,0.385902,0.148920


### Evaluate the H2O AutoML Leader on the Hold-Out Test Set

The H2O AutoML leaderboard ranks models using cross-validation performance. The selected leader is therefore evaluated separately on the hold-out test set.

This provides a test-set AUC that can be compared with the tuned PyCaret model's hold-out ROC-AUC of approximately **0.8296**.

In [70]:
# Get the H2O AutoML leader
h2o_leader = h2o_aml.leader

print("H2O AutoML Leader:")
print(h2o_leader.model_id)

# Predict on the hold-out test set
h2o_test_predictions = h2o_leader.predict(test_h2o)

# Convert H2O predictions to pandas
h2o_pred_df = h2o_test_predictions.as_data_frame()

# H2O binary classification returns p0 and p1
h2o_test_probability = h2o_pred_df['p1'].values

# Actual test labels
h2o_test_actual = test_h2o['Churn'].as_data_frame().iloc[:, 0].astype(int).values

# Calculate test ROC-AUC
h2o_test_auc = roc_auc_score(
    h2o_test_actual,
    h2o_test_probability
)

print(f"H2O AutoML hold-out ROC-AUC: {h2o_test_auc:.4f}")

H2O AutoML Leader:
StackedEnsemble_BestOfFamily_1_AutoML_2_20260928_110138
stackedensemble prediction progress: |███████████████████████████████████████████| (done) 100%
H2O AutoML hold-out ROC-AUC: 0.8319


### PyCaret vs H2O AutoML Comparison

The two AutoML frameworks can now be compared using the same primary metric:

- **PyCaret tuned model hold-out ROC-AUC:** approximately 0.8296
- **H2O AutoML leader hold-out ROC-AUC:** calculated above

This comparison shows whether the second AutoML framework produces a model with similar or different discriminatory performance.

The comparison is descriptive rather than based on a single metric alone. Differences may also arise from the algorithms searched, preprocessing methods, hyperparameter spaces, and ensemble strategies used by each framework.

In [71]:
# PyCaret hold-out ROC-AUC from the previous evaluation
pycaret_test_auc = roc_auc_score(y_test, y_proba)

comparison_df = pd.DataFrame({
    'AutoML Framework': [
        'PyCaret',
        'H2O AutoML'
    ],
    'Selected Model': [
        'Tuned Gradient Boosting Classifier',
        h2o_leader.model_id
    ],
    'Hold-out ROC-AUC': [
        pycaret_test_auc,
        h2o_test_auc
    ]
})

comparison_df

,AutoML Framework,Selected Model,Hold-out ROC-AUC
0,PyCaret,Tuned Gradient Boosting Classifier,0.829559
1,H2O AutoML,StackedEnsemble_BestOfFamily_1_AutoML_2_202609...,0.831934


### Feature Importance Comparison

Feature importance provides another way to compare the automated models.

The PyCaret workflow used the engineered churn variables from Week 2, while H2O AutoML automatically evaluates models using the same prepared feature set.

If the H2O AutoML leader supports variable importance, its most influential features are displayed below. If the leader is an ensemble without direct variable importance, the notebook reports that limitation rather than substituting an unrelated measure.

In [72]:
# Display H2O leader feature importance when supported
try:
    h2o_varimp = h2o_leader.varimp(use_pandas=True)

    print("H2O AutoML Leader Feature Importance:")
    display(h2o_varimp.head(15))

except Exception as e:
    print(
        "Direct feature importance is not available for the H2O AutoML "
        "leader model."
    )
    print("Reason:", str(e))

H2O AutoML Leader Feature Importance:
Direct feature importance is not available for the H2O AutoML leader model.
Reason: 'NoneType' object has no attribute 'head'


## Additional Challenge: User-Specified Input File

The `ChurnPredictor` module is extended so that users can specify the input CSV file at runtime.

The module supports two ways of providing data:

1. Calling `predict_file()` from Python with a specified file path.
2. Running `churn_predictor.py` directly and entering the file path when prompted.

If the user presses Enter without specifying a path, the module uses `new_churn_data.csv` as the default file.

### Test User-Specified File Input

The `predict_file()` method allows a user to specify any compatible CSV file path programmatically.

The following example uses `new_churn_data.csv`. When the Python module is executed directly from the command line, the same functionality is available through the interactive `input()` prompt.

In [73]:
# Test the user-file functionality without requiring interactive input
import importlib
import churn_predictor

importlib.reload(churn_predictor)

from churn_predictor import ChurnPredictor

predictor = ChurnPredictor('churn_model.joblib')

# Training probabilities for percentile calculation
train_probs = tuned_model.pipeline.predict_proba(
    exp.X_train
)[:, 1]

predictor.set_training_probs(train_probs)

# User-specified file
user_file = 'new_churn_data.csv'

user_results = predictor.predict_file(user_file)

predictor.print_predictions_from_results(user_results)

display(user_results)

Loaded file: new_churn_data.csv
Rows: 5
Columns: 8

Prediction Results:
   Churn_Probability  Churn_Prediction
0           0.451574                 0
1           0.189215                 0
2           0.043248                 0
3           0.253950                 0
4           0.274610                 0

Churn Percentiles:
   Churn_Probability  Churn_Percentile
0           0.451574         76.782222
1           0.189215         51.431111
2           0.043248         21.920000
3           0.253950         58.275556
4           0.274610         60.106667


,customerID,tenure,PhoneService,Contract,PaymentMethod,MonthlyCharges,TotalCharges,charge_per_tenure,Churn_Probability,Churn_Prediction,Churn_Percentile
0,9305-CKSKC,22,1,0,2,97.40,811.70,36.895455,0.451574,0,76.782222
1,1452-KNGVK,8,0,1,1,77.30,1701.95,212.743750,0.189215,0,51.431111
2,6723-OKKJM,28,1,0,0,28.25,250.90,8.960714,0.043248,0,21.920000
3,7832-POPKP,62,1,0,2,101.70,3106.56,50.105806,0.253950,0,58.275556
4,6348-TACGU,10,0,0,1,51.15,3440.97,344.097000,0.274610,0,60.106667


## Additional Challenge: Use the Unmodified Week 1 Churn Data

The unmodified Week 1 dataset is processed through the same major preprocessing steps used during Week 2 before being passed to the trained model.

The preprocessing includes:

1. Converting `TotalCharges` to numeric values.
2. Handling missing values created by invalid or blank `TotalCharges` entries.
3. Creating `AvgMonthlyCharge`.
4. Creating `TenureMonthlyRatio`.
5. Creating `LifetimeValue`.
6. Creating `TenureBin`.
7. Creating `ChargeBin`.
8. Encoding the categorical variables using the same mappings used during Week 2.
9. Removing `customerID`.
10. Aligning the resulting columns with the model's expected feature schema.

The target `Churn` is not required for prediction and is removed if it is present.

## Additional Challenge: Use the Unmodified Week 1 Churn Data

The unmodified Week 1 dataset is processed through the same major preprocessing steps used during Week 2 before being passed to the trained model.

The preprocessing includes:

1. Converting `TotalCharges` to numeric values.
2. Handling missing values created by invalid or blank `TotalCharges` entries.
3. Creating `AvgMonthlyCharge`.
4. Creating `TenureMonthlyRatio`.
5. Creating `LifetimeValue`.
6. Creating `TenureBin`.
7. Creating `ChargeBin`.
8. Encoding the categorical variables using the same mappings used during Week 2.
9. Removing `customerID`.
10. Aligning the resulting columns with the model's expected feature schema.

The target `Churn` is not required for prediction and is removed if it is present.

In [74]:
def prepare_unmodified_week1_data(df):
    """
    Prepare unmodified Week 1 churn data using the
    preprocessing approach used in Week 2.
    """

    df = df.copy()

    # --------------------------------------------------
    # 1. Convert TotalCharges to numeric
    # --------------------------------------------------

    df['TotalCharges'] = pd.to_numeric(
        df['TotalCharges'],
        errors='coerce'
    )

    # --------------------------------------------------
    # 2. Remove rows with missing TotalCharges
    # --------------------------------------------------

    before_rows = len(df)

    df = df.dropna(
        subset=['TotalCharges']
    ).copy()

    removed_rows = before_rows - len(df)

    print(
        f"Rows removed because of invalid/missing "
        f"TotalCharges: {removed_rows}"
    )

    # --------------------------------------------------
    # 3. Create engineered numerical features
    # --------------------------------------------------

    df['AvgMonthlyCharge'] = (
        df['TotalCharges'] /
        df['tenure'].replace(0, np.nan)
    )

    df['TenureMonthlyRatio'] = (
        df['tenure'] /
        df['MonthlyCharges'].replace(0, np.nan)
    )

    df['LifetimeValue'] = (
        df['MonthlyCharges'] *
        df['tenure']
    )

    # --------------------------------------------------
    # 4. Create TenureBin
    # --------------------------------------------------

    df['TenureBin'] = pd.cut(
        df['tenure'],
        bins=[-1, 6, 12, 24, 48, 72],
        labels=[
            '0-6',
            '6-12',
            '12-24',
            '24-48',
            '48-72'
        ]
    )

    # --------------------------------------------------
    # 5. Create ChargeBin
    # --------------------------------------------------

    df['ChargeBin'] = pd.cut(
        df['MonthlyCharges'],
        bins=[0, 30, 50, 70, 90, 120],
        labels=[
            '0-30',
            '30-50',
            '50-70',
            '70-90',
            '90-120'
        ]
    )

    # --------------------------------------------------
    # 6. Encode PhoneService
    # --------------------------------------------------

    phone_map = {
        'No': 0,
        'Yes': 1
    }

    if df['PhoneService'].dtype == 'object':
        df['PhoneService'] = (
            df['PhoneService']
            .map(phone_map)
        )

    # --------------------------------------------------
    # 7. Encode Contract
    # --------------------------------------------------

    contract_map = {
        'Month-to-month': 0,
        'One year': 1,
        'Two year': 2
    }

    if df['Contract'].dtype == 'object':
        df['Contract'] = (
            df['Contract']
            .map(contract_map)
        )

    # --------------------------------------------------
    # 8. Encode PaymentMethod
    # --------------------------------------------------

    payment_map = {
        'Bank transfer (automatic)': 0,
        'Credit card (automatic)': 1,
        'Electronic check': 2,
        'Mailed check': 3
    }

    if df['PaymentMethod'].dtype == 'object':
        df['PaymentMethod'] = (
            df['PaymentMethod']
            .map(payment_map)
        )

    # --------------------------------------------------
    # 9. Encode Churn if present
    # --------------------------------------------------

    churn_map = {
        'No': 0,
        'Yes': 1
    }

    if 'Churn' in df.columns:
        if df['Churn'].dtype == 'object':
            df['Churn'] = (
                df['Churn']
                .map(churn_map)
            )

    # --------------------------------------------------
    # 10. Remove customerID
    # --------------------------------------------------

    if 'customerID' in df.columns:
        df = df.drop(
            columns=['customerID']
        )

    # --------------------------------------------------
    # 11. Remove any feature not used by the model
    # --------------------------------------------------

    if 'charge_per_tenure' in df.columns:
        df = df.drop(
            columns=['charge_per_tenure']
        )

    # --------------------------------------------------
    # 12. Remove target from prediction features
    # --------------------------------------------------

    if 'Churn' in df.columns:
        df = df.drop(
            columns=['Churn']
        )

    # --------------------------------------------------
    # 13. Match model feature schema
    # --------------------------------------------------

    expected_cols = [
        'tenure',
        'PhoneService',
        'Contract',
        'PaymentMethod',
        'MonthlyCharges',
        'TotalCharges',
        'AvgMonthlyCharge',
        'TenureMonthlyRatio',
        'LifetimeValue',
        'TenureBin',
        'ChargeBin'
    ]

    missing_cols = [
        col for col in expected_cols
        if col not in df.columns
    ]

    if missing_cols:
        raise ValueError(
            f"Missing required columns: {missing_cols}"
        )

    df = df[expected_cols]

    # --------------------------------------------------
    # 14. Check for remaining missing values
    # --------------------------------------------------

    if df.isnull().any().any():
        print(
            "\\nWarning: Missing values remain after "
            "preprocessing:"
        )
        print(
            df.isnull().sum()[
                df.isnull().sum() > 0
            ]
        )

    return df

### Load the Unmodified Week 1 Dataset

The unmodified churn dataset is loaded directly from `new_unmodified_churn_data.csv`.

Unlike `new_churn_data.csv`, this dataset follows the original Week 1 data structure and therefore requires the categorical encoding and data-cleaning steps that were performed during Week 2.

In [75]:
# Load the unmodified Week 1 churn data
unmodified_data = pd.read_csv(
    'new_churn_data_unmodified.csv'
)

print(
    "Unmodified Week 1 data shape:",
    unmodified_data.shape
)

print("\nOriginal columns:")
print(
    unmodified_data.columns.tolist()
)

print("\nFirst five rows:")
display(
    unmodified_data.head()
)

Unmodified Week 1 data shape: (5, 7)

Original columns:
['customerID', 'tenure', 'PhoneService', 'Contract', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']

First five rows:


,customerID,tenure,PhoneService,Contract,PaymentMethod,MonthlyCharges,TotalCharges
0,9305-CKSKC,22,Yes,Month-to-month,Electronic check,97.40,811.70
1,1452-KNGVK,8,No,One year,Mailed check,77.30,1701.95
2,6723-OKKJM,28,Yes,Month-to-month,Credit card (automatic),28.25,250.90
3,7832-POPKP,62,Yes,Month-to-month,Electronic check,101.70,3106.56
4,6348-TACGU,10,No,Two year,Credit card (automatic),51.15,3440.97


In [76]:
# Apply the Week 2 preprocessing workflow
unmodified_features = prepare_unmodified_week1_data(
    unmodified_data
)

print(
    "\nPreprocessed Week 1 data shape:",
    unmodified_features.shape
)

print("\nModel input columns:")
print(
    unmodified_features.columns.tolist()
)

print("\nPreprocessed data:")
display(
    unmodified_features.head()
)

Rows removed because of invalid/missing TotalCharges: 0

Preprocessed Week 1 data shape: (5, 11)

Model input columns:
['tenure', 'PhoneService', 'Contract', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlyCharge', 'TenureMonthlyRatio', 'LifetimeValue', 'TenureBin', 'ChargeBin']

Preprocessed data:


,tenure,PhoneService,Contract,PaymentMethod,MonthlyCharges,TotalCharges,AvgMonthlyCharge,TenureMonthlyRatio,LifetimeValue,TenureBin,ChargeBin
0,22,1,0,2,97.40,811.70,36.895455,0.225873,2142.8,12-24,90-120
1,8,0,1,3,77.30,1701.95,212.743750,0.103493,618.4,6-12,70-90
2,28,1,0,1,28.25,250.90,8.960714,0.991150,791.0,24-48,0-30
3,62,1,0,2,101.70,3106.56,50.105806,0.609636,6305.4,48-72,90-120
4,10,0,2,1,51.15,3440.97,344.097000,0.195503,511.5,6-12,50-70


### Generate Predictions from the Unmodified Week 1 Data

After preprocessing, the unmodified Week 1 data has been transformed to match the feature schema expected by the trained Week 5 model.

The saved model is then used to generate:

- Churn probability
- Binary churn prediction
- Churn probability percentile relative to the training probability distribution

In [77]:
# Initialize the predictor
week1_predictor = ChurnPredictor(
    'churn_model.joblib'
)

# Calculate training probabilities for percentile calculation
training_probabilities = tuned_model.pipeline.predict_proba(
    exp.X_train
)[:, 1]

week1_predictor.set_training_probs(
    training_probabilities
)

# Generate predictions using already-prepared features
week1_results = week1_predictor.predict(
    unmodified_features,
    prepare=False
)

print("Predictions for unmodified Week 1 data:")
display(
    week1_results[
        [
            'Churn_Probability',
            'Churn_Prediction',
            'Churn_Percentile'
        ]
    ]
)

Predictions for unmodified Week 1 data:


,Churn_Probability,Churn_Prediction,Churn_Percentile
0,0.451574,0,76.782222
1,0.176632,0,49.991111
2,0.043248,0,21.920000
3,0.253950,0,58.275556
4,0.060725,0,26.968889


### Verification of the Unmodified Week 1 Data Challenge

The unmodified Week 1 dataset was successfully processed using the Week 2 preprocessing logic.

The resulting data was aligned with the feature schema used by the trained model, allowing the saved Week 5 model to generate churn probabilities and predictions.

This completes the optional **Unmodified Week 1 Data** challenge.